In [1]:
!pip install stable_baselines3
!pip install gymnasium[atari,accept-rom-license]
!pip install ale-py

zsh:1: no matches found: gymnasium[atari,accept-rom-license]


In [2]:
import os
import random
import time

import ale_py
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from stable_baselines3.common.buffers import RolloutBuffer
from stable_baselines3.common.atari_wrappers import (
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)

In [3]:
# settings
env_id  = "BreakoutNoFrameskip-v4"

# number of environments to run in parallel
num_envs = 16

# Number of steps to collect per environment before update (N)
n_steps = 5

# total number of timesteps for training over multiple episodes
total_timesteps = 10_000_000

# adam learning rate
learning_rate = 2e-4

# discount factor
gamma = 0.99

# GAE lambda parameter
gae_lambda = 0.95 # Common default value for GAE

# optional: set seed for reproducibility
# seed = 123
seed = None

# entropy coefficient for loss
entropy_weight = 0.01

# value function coefficient for loss
value_weight = 0.25

# max L2-norm for gradient clipping
max_norm = 0.5

# where to save videos
video_path = "videos_a2c_gae_atari"

In [4]:
# will encapsulate in lambda later
def make_env(env_id, capture_video, seed=None):
    if capture_video:
        env = gym.make(env_id, render_mode="rgb_array")
        env = gym.wrappers.RecordVideo(
            env, video_path, episode_trigger=lambda episode_id: True)
    else:
        env = gym.make(env_id)
    env = gym.wrappers.RecordEpisodeStatistics(env) # Records episode returns and lengths

    env = NoopResetEnv(env, noop_max=30)
    env = MaxAndSkipEnv(env, skip=4)
    env = EpisodicLifeEnv(env)
    if "FIRE" in env.unwrapped.get_action_meanings():
        env = FireResetEnv(env)
    env = ClipRewardEnv(env)
    env = gym.wrappers.ResizeObservation(env, (84, 84))
    env = gym.wrappers.GrayscaleObservation(env)
    env = gym.wrappers.FrameStackObservation(env, 4)

    # To get reproducible sampling of actions, a seed can be set with env.action_space.seed(123)
    # Note: For vector envs, seeding is often handled during initialization
    if seed is not None:
        env.action_space.seed(seed)
        # Important: Also seed the observation space for reproducibility if necessary
        # env.observation_space.seed(seed)

    return env

In [5]:
class ActorCritic(nn.Module):
  def __init__(self, envs):
    super().__init__()

    n_input_channels = envs.single_observation_space.shape[0]
    self.cnn = nn.Sequential(
        nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4, padding=0),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=0),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=0),
        nn.ReLU(),
        nn.Flatten(),
    )

    # Compute shape by doing one forward pass
    with torch.no_grad():
      n_flatten = self.cnn(
          torch.as_tensor(
              envs.single_observation_space.sample()[None]).float()).shape[1]

    self.linear = nn.Sequential(
        nn.Linear(n_flatten, 512),
        nn.ReLU(),
    )

    # Actor (Policy) Head
    self.actor = nn.Linear(512, envs.single_action_space.n)

    # Critic (Value Function) Head
    self.critic = nn.Linear(512, 1)

  def forward(self, x):
    features = self.cnn(x / 255.0)
    shared_latent = self.linear(features)

    return self.actor(shared_latent), self.critic(shared_latent)

In [6]:
# make environments
envs = gym.vector.SyncVectorEnv([
    lambda: make_env(env_id, False, seed + i if seed is not None else None)
    for i in range(num_envs)
])

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]


In [7]:
# optional: set seeds for reproducibility
if seed is not None:
  torch.manual_seed(seed)
  random.seed(seed)
  np.random.seed(seed)
  if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

In [8]:
# set device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# On Mac, MPS faster than CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [9]:
# make neural network
ac_network = ActorCritic(envs).to(device)
optimizer = optim.Adam(ac_network.parameters(), lr=learning_rate, eps=1e-5)

In [10]:
rollout_buffer = RolloutBuffer(
    buffer_size=n_steps,
    observation_space=envs.single_observation_space,
    action_space=envs.single_action_space,
    device=device,
    gamma=gamma,
    gae_lambda=gae_lambda,
    n_envs=num_envs,
)

In [11]:
# sample an action from logits
def sample_action(logits):
  distribution = Categorical(logits=logits)
  return distribution.sample()

In [12]:
# compute entropy for loss regularization
def compute_entropy_and_log_probs(policy_logits, actions):
  distribution = Categorical(logits=policy_logits)
  entropy = distribution.entropy()
  log_probs = distribution.log_prob(actions)
  return entropy.mean(), log_probs

In [13]:
def np2torch(a, dtype=torch.float32, device=device):
  return torch.as_tensor(a, dtype=dtype, device=device)

In [14]:
# training loop
episode_returns = []
losses = []

In [15]:
global_step = 0
start_time = time.time()

# rollout buffer uses start flag, not terminated flag
is_start = np.ones(num_envs)

# reset env
obs, _ = envs.reset(seed=seed)

# Main training loop
while global_step < total_timesteps:

  # Rollout phase
  for step in range(n_steps):
    # Convert obs to tensor
    obs_tensor = np2torch(obs)

    # Forward pass through the network
    with torch.no_grad():
      policy_logits, values = ac_network(obs_tensor)
      actions = sample_action(policy_logits)
      _, log_probs = compute_entropy_and_log_probs(policy_logits, actions)

    # Take actions in the environment
    actions_np = actions.cpu().numpy()
    next_obs, rewards, terminateds, truncateds, infos = envs.step(actions_np)
    dones = terminateds | truncateds

    # Store data in buffer
    rollout_buffer.add(
        obs,
        actions_np,
        rewards,
        is_start,
        values.squeeze(-1),
        log_probs,
    )

    # Update observations
    obs = next_obs

    # Update start flag for the NEXT round
    is_start = dones

    for i, (terminated, truncated) in enumerate(zip(terminateds, truncateds)):
      if terminated or truncated:
        # This only appears when all lives are lost
        if 'episode' in infos:
          ret = infos['episode']['r'][i]
          episode_returns.append(ret)
          print(f"global_step={global_step}, episode={len(episode_returns)}, episode_return={ret}")

  # Update phase
  with torch.no_grad():
    _, last_values = ac_network(np2torch(next_obs))
    last_dones = dones

  rollout_buffer.compute_returns_and_advantage(
      last_values=last_values.squeeze(-1), dones=last_dones)

  # This will only loop once (get all data in one go)
  for rollout_data in rollout_buffer.get(batch_size=None):

    observations_arr = rollout_data.observations
    actions_arr = rollout_data.actions.flatten()
    advantages_arr = rollout_data.advantages
    returns_arr = rollout_data.returns

    # Calculate
    new_logits, new_values = ac_network(observations_arr)
    entropy, new_log_probs = compute_entropy_and_log_probs(
        new_logits, actions_arr)
    new_values = new_values.flatten()

    # Value Loss (MSE)
    value_loss = F.mse_loss(new_values, returns_arr)

    # Polciy Loss
    policy_loss = -(advantages_arr * new_log_probs).mean()

    # Entropy Loss
    entropy_loss = -entropy

    # Total Loss
    loss = policy_loss + value_weight * value_loss + entropy_weight * entropy_loss

    # Store losses for plotting
    losses.append(loss.item())

    # Update network
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(ac_network.parameters(), max_norm)
    optimizer.step()

  # Important: reset the buffer
  # If you don't do this, observations will have the wrong shape!
  # It flattens from (N, num_envs, 4, 84, 84) --> (N * num_envs, 4, 84, 84)
  rollout_buffer.reset()

  # Update global step counter
  global_step += n_steps * num_envs

  # Logging
  if global_step % (100 * n_steps * num_envs) == 0 and len(episode_returns) > 100: # Log approx every 100 updates
    print(f"Global Step: {global_step} / {total_timesteps}")
    print(f"  Loss: {loss.item():.4f} (Policy: {policy_loss.item():.4f}, Value: {value_loss.item():.4f}, Entropy: {-entropy_loss.item():.4f})") # print positive entropy
    if episode_returns:
        print(f"  Mean Return (last 100): {np.mean(episode_returns[-100:]):.2f}")
    print(f"  Steps per second: {int(global_step / (time.time() - start_time))}")

# close
envs.close()

global_step=1760, episode=1, episode_return=0.0
global_step=1760, episode=2, episode_return=0.0
global_step=1760, episode=3, episode_return=0.0
global_step=1840, episode=4, episode_return=0.0
global_step=1840, episode=5, episode_return=0.0
global_step=1840, episode=6, episode_return=0.0
global_step=1840, episode=7, episode_return=0.0
global_step=2560, episode=8, episode_return=1.0
global_step=2560, episode=9, episode_return=1.0
global_step=2560, episode=10, episode_return=0.0
global_step=2640, episode=11, episode_return=1.0
global_step=3280, episode=12, episode_return=2.0
global_step=3280, episode=13, episode_return=2.0
global_step=3360, episode=14, episode_return=0.0
global_step=3360, episode=15, episode_return=2.0
global_step=3680, episode=16, episode_return=0.0
global_step=3760, episode=17, episode_return=0.0
global_step=3840, episode=18, episode_return=3.0
global_step=4400, episode=19, episode_return=1.0
global_step=4400, episode=20, episode_return=1.0
global_step=4480, episode=21,

KeyboardInterrupt: 

In [ ]:
def smooth(x, a=0.1):
    y = [x[0]]
    for xi in x[1:]:
        yi = a * xi + (1 - a) * y[-1]
        y.append(yi)
    return y

In [ ]:
plt.plot(episode_returns, alpha=0.2)
plt.plot(smooth(episode_returns))
plt.title("episode_returns");

In [ ]:
plt.plot(losses)
plt.title("losses");

In [ ]:
# --- Save Model ---
model_path = f"a2c_gae_atari_{env_id}.pth"
print(f"Saving model to {model_path}")
torch.save(ac_network.state_dict(), model_path)

In [ ]:
# --- Evaluation Phase ---

# load model for eval
envs_eval = gym.vector.SyncVectorEnv([lambda: make_env(env_id, True)],
    autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
model = ActorCritic(envs_eval).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

In [ ]:
# evaluate the model
n_episodes_eval = 10
eval_returns = np.zeros(n_episodes_eval)
obs, _ = envs_eval.reset()
for i in range(n_episodes_eval):
  episode_done = False
  while not episode_done:
    with torch.no_grad():
      logits, _ = model(np2torch(obs))
      actions = sample_action(logits)
      actions = actions.cpu().numpy()
      obs, rewards, terminateds, truncateds, infos = envs_eval.step(actions)
      if terminateds[0] or truncateds[0]:
        if 'final_info' in infos:
          if 'episode' in infos['final_info']:
            episode_done = True
            ret = infos['final_info']["episode"]["r"][0]
            print(f"episode={i}, return={ret}")
            eval_returns[i] = ret
envs_eval.close()

In [ ]:
# plot the eval returns distribution
plt.hist(eval_returns)
plt.title("Eval Returns")
plt.show();

![](https://deeplearningcourses.com/notebooks_v3_pxl?sc=83SBIReT_VK9g4jXKsbP_w&n=A2C+Atari)